In [ ]:
!pip install requests pandas pyarrow

In [21]:
import requests
import pandas as pd
from datetime import datetime
import pyarrow

In [45]:
BASE_URL = "https://opendata.paris.fr/api/records/1.0/search/"

params = {
    "dataset": "velib-disponibilite-en-temps-reel",
    "rows": 1506,  # max pour exploration
}

response = requests.get(BASE_URL, params=params)
data = response.json()

records = data["records"]

rows = []
for r in records:
    row = r["fields"]
    row["record_timestamp"] = r["record_timestamp"]
    rows.append(row)

df = pd.DataFrame(rows)
len(df.index)

1506

In [67]:
API_URL = "https://opendata.paris.fr/api/explore/v2.1/catalog/datasets/velib-disponibilite-en-temps-reel/records"
params = {"limit": -1}
raw = requests.get(API_URL, params=params, timeout=30)
raw.raise_for_status()
response_json = raw.json()
# Verifying keys
if "results" in response_json:
    data = response_json["results"]
elif "records" in response_json:
    data = response_json["records"]
else:
    # Save analysis for debugging
    debug_path = "/app/data_lake/debug"
    os.makedirs(debug_path, exist_ok=True)
    with open(f"{debug_path}/api_response_{timestamp}.json", "w") as f:
        json.dump(response_json, f, indent=2)
    sys.exit(1)
if not data:
    sys.exit(1)
# Create DataFrame
dataframe = pd.DataFrame(data)
len(dataframe.index)

100

In [58]:
API_URL = "https://opendata.paris.fr/api/explore/v2.1/catalog/datasets/velib-disponibilite-en-temps-reel/exports/parquet?parquet_compression=snappy"
pq = requests.get(API_URL, timeout=30)

In [70]:
API_URL = "https://opendata.paris.fr/api/explore/v2.1/catalog/datasets/velib-disponibilite-en-temps-reel/exports/csv?delimiter=%3B&list_separator=%2C&quote_all=false&with_bom=true"
raw = requests.get(API_URL, timeout=30)
print(pq.status_code)

200


In [73]:
raw_csv = raw.json()

JSONDecodeError: Unexpected UTF-8 BOM (decode using utf-8-sig): line 1 column 1 (char 0)

In [66]:
print(pq.status_code)
file_size = pq.stat().st_size / 1024  # en KB
print(f"{i}. {file.name} ({file_size:.2f} KB)")

200


AttributeError: 'Response' object has no attribute 'stat'

In [40]:
df.head()

,name,stationcode,ebike,mechanical,coordonnees_geo,duedate,numbikesavailable,numdocksavailable,capacity,is_renting,is_installed,nom_arrondissement_communes,is_returning,code_insee_commune,record_timestamp
0,Charcot - Benfleet,32304,6,3,"[48.878370277021, 2.440523876268]",2026-02-11T19:13:29+00:00,9,19,28,OUI,OUI,Romainville,OUI,93063,2026-02-11T19:50:00.271Z
1,Messine - Place Du Pérou,8026,5,5,"[48.875448033960744, 2.315508019010038]",2026-02-11T19:13:03+00:00,10,2,12,OUI,OUI,Paris,OUI,75056,2026-02-11T19:50:00.271Z
2,Saint-Romain - Cherche-Midi,6108,3,12,"[48.84708159081946, 2.321374788880348]",2026-02-11T19:13:03+00:00,15,1,17,OUI,OUI,Paris,OUI,75056,2026-02-11T19:50:00.271Z
3,André Karman - République,33006,8,7,"[48.91039875761846, 2.3851355910301213]",2026-02-11T19:15:17+00:00,15,10,31,OUI,OUI,Aubervilliers,OUI,93001,2026-02-11T19:50:00.271Z
4,Pierre et Marie Curie - Maurice Thorez,42016,4,0,"[48.81580226360801, 2.376804985105991]",2026-02-11T19:15:00+00:00,4,21,27,OUI,OUI,Ivry-sur-Seine,OUI,94041,2026-02-11T19:50:00.271Z


In [41]:
df.columns

Index(['name', 'stationcode', 'ebike', 'mechanical', 'coordonnees_geo',
       'duedate', 'numbikesavailable', 'numdocksavailable', 'capacity',
       'is_renting', 'is_installed', 'nom_arrondissement_communes',
       'is_returning', 'code_insee_commune', 'record_timestamp'],
      dtype='object')

In [42]:
len(df.index)

1506

In [20]:
df["record_timestamp"] = pd.to_datetime(df["record_timestamp"])

df["availability_ratio"] = df["numbikesavailable"] / df["capacity"]

df[[
    "stationcode",
    "name",
    "nom_arrondissement_communes",
    "capacity",
    "numbikesavailable",
    "numdocksavailable",
    "availability_ratio",
    "record_timestamp"
]].head()

,stationcode,name,nom_arrondissement_communes,capacity,numbikesavailable,numdocksavailable,availability_ratio,record_timestamp
0,16107,Benjamin Godard - Victor Hugo,Paris,35,18,17,0.514286,2026-02-11 10:34:00.258000+00:00
1,40001,Hôpital Mondor,Créteil,28,16,12,0.571429,2026-02-11 10:34:00.258000+00:00
2,14014,Jourdan - Stade Charléty,Paris,60,7,51,0.116667,2026-02-11 10:34:00.258000+00:00
3,32017,Basilique,Saint-Denis,22,19,2,0.863636,2026-02-11 10:34:00.258000+00:00
4,8026,Messine - Place Du Pérou,Paris,12,11,1,0.916667,2026-02-11 10:34:00.258000+00:00


In [8]:
df.sort_values("availability_ratio").head(10)[[
    "name", "nom_arrondissement_communes", "capacity",
    "numbikesavailable", "availability_ratio"
]]

,name,nom_arrondissement_communes,capacity,numbikesavailable,availability_ratio
968,Vladimir Ilitch Lénine - Andrée Chedid,Gentilly,31,0,0.0
886,Blaise-Desgoffe - Vaugirard,Paris,32,0,0.0
1016,Lepic - Armée d'Orient,Paris,24,0,0.0
745,Réaumur - Montmartre,Paris,38,0,0.0
180,Gravelle - Route du Bac,Paris,51,0,0.0
1213,Condorcet - Turgot,Paris,20,0,0.0
1113,Versailles - Résistances,Thiais,1,0,0.0
159,Gare Arcueil - Cachan,Arcueil,26,0,0.0
1117,Demi-Lune - Aristide Briand,Montreuil,20,0,0.0
1205,Champs-Elysees - Bassano,Paris,25,0,0.0


In [9]:
df.groupby("nom_arrondissement_communes").agg({
    "numbikesavailable": "sum",
    "capacity": "sum"
}).assign(
    availability_ratio=lambda x: x["numbikesavailable"] / x["capacity"]
).sort_values("availability_ratio")

,numbikesavailable,capacity,availability_ratio
nom_arrondissement_communes,,,
Ville-d'Avray,0,24,0.000000
Le Pré-Saint-Gervais,3,42,0.071429
Châtillon,15,177,0.084746
Le Kremlin-Bicêtre,15,149,0.100671
Montrouge,37,354,0.104520
...,...,...,...
Bourg-la-Reine,47,77,0.610390
Noisy-le-Sec,40,58,0.689655
Levallois-Perret,255,360,0.708333


In [10]:
df["hour"] = df["record_timestamp"].dt.hour

df.groupby("hour")["availability_ratio"].mean()

hour
10    0.403194
Name: availability_ratio, dtype: float64

In [74]:
import requests
import pandas as pd
from io import BytesIO

# URL de l'API
url = "https://opendata.paris.fr/api/explore/v2.1/catalog/datasets/velib-disponibilite-en-temps-reel/exports/parquet"

# Télécharger
response = requests.get(url, params={'parquet_compression': 'snappy'})

# Afficher infos
print(f"Status: {response.status_code}")
print(f"Content-Type: {response.headers.get('Content-Type')}")
print(f"Taille: {len(response.content) / 1024:.2f} KB")

# Charger dans pandas
df = pd.read_parquet(BytesIO(response.content))

# Explorer
print(f"\nShape: {df.shape}")
print(f"\nColonnes:\n{df.columns.tolist()}")
print(f"\nPremières lignes:")
df.head()

Status: 200
Content-Type: application/parquet; charset=utf-8
Taille: 81.63 KB

Shape: (1506, 15)

Colonnes:
['stationcode', 'name', 'is_installed', 'capacity', 'numdocksavailable', 'numbikesavailable', 'mechanical', 'ebike', 'is_renting', 'is_returning', 'duedate', 'coordonnees_geo', 'nom_arrondissement_communes', 'code_insee_commune', 'station_opening_hours']

Premières lignes:


,stationcode,name,is_installed,capacity,numdocksavailable,numbikesavailable,mechanical,ebike,is_renting,is_returning,duedate,coordonnees_geo,nom_arrondissement_communes,code_insee_commune,station_opening_hours
0,16107,Benjamin Godard - Victor Hugo,OUI,35,23,11,4,7,OUI,OUI,2026-02-11 20:11:53+00:00,"b'\x01\x01\x00\x00\x00M\x84\rO\xaf4\x02@,\xf2\...",Paris,75056,None
1,40001,Hôpital Mondor,OUI,28,10,18,10,8,OUI,OUI,2026-02-11 20:13:35+00:00,b'\x01\x01\x00\x00\x00\xbcI\x8b#E\xa1\x03@\xce...,Créteil,94028,None
2,32304,Charcot - Benfleet,OUI,28,18,9,3,6,OUI,OUI,2026-02-11 20:13:38+00:00,b'\x01\x01\x00\x00\x00\xfal\xcda1\x86\x03@\xb5...,Romainville,93063,None
3,9020,Toudouze - Clauzel,OUI,21,13,6,3,3,OUI,OUI,2026-02-11 20:06:48+00:00,b'\x01\x01\x00\x00\x00\x01\x00\x00\xd8\xe9\xb2...,Paris,75056,None
4,14111,Cassini - Denfert-Rochereau,OUI,25,20,4,1,3,OUI,OUI,2026-02-11 20:11:36+00:00,b'\x01\x01\x00\x00\x00\xca\xff\xffT3\xb0\x02@\...,Paris,75056,None
